In [1]:
import os

In [2]:
%pwd

'd:\\MachineLearning\\END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\MachineLearning\\END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW'

In [5]:
os.environ["MFLOW_TRACKING_URI"]="https://dagshub.com/Faraz-Gill/END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="FARAZ-GILL"
os.environ["MLFLOW_TRACKING_PASSWORD"]="9e32b96a9a5044303f6832c7d3038ee75105e088"

In [29]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [30]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [31]:
class ConfigurationManager:
    def __init__ (
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema,
            mlflow_uri="https://dagshub.com/Faraz-Gill/END-TO-ENT-MACHINE-LEARNING-PROJECT-WITH-MLFLOW.mlflow"
        )

        return model_evaluation_config

In [32]:
import os
from urllib.parse import urlparse
import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [33]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2


    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        mlflow.set_tracking_uri(self.config.mlflow_uri)
        mlflow.set_tracking_uri(self.config.mlflow_uri)

        tracking_url_type_store = urlparse(
            mlflow.get_tracking_uri()
        ).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            # Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metrics({
                "rmse": rmse,
                "mae": mae,
                "r2": r2
            })

            # Model registry does not  work with file store
            if tracking_url_type_store != "file":

                # Register the model
                # There are other ways to use the Model Registry, which depends on the ue case,
                # please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

In [34]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()
except Exception as e:
    raise


[2026-11-27 14:56:11,333: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-11-27 14:56:11,336: INFO: common: yaml file: params.yaml loaded successfully]
[2026-11-27 14:56:11,343: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-11-27 14:56:11,344: INFO: common: created directory at: artifacts]
[2026-11-27 14:56:11,346: INFO: common: created directory at: artifacts/model_evaluation]
[2026-11-27 14:56:12,580: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


c:\Users\Rehan\anaconda3\envs\mlproj\lib\site-packages\_distutils_hack\__init__.py:11: UserWarning: Distutils was imported before Setuptools, but importing Setuptools also replaces the `distutils` module in `sys.modules`. This may lead to undesirable behaviors or errors. To avoid these issues, avoid using distutils directly, ensure that setuptools is installed in the traditional way (e.g. not an editable install), and/or make sure that setuptools is always imported before distutils.
  warnings.warn(
c:\Users\Rehan\anaconda3\envs\mlproj\lib\site-packages\_distutils_hack\__init__.py:26: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/11/27 14:56:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetMod